In [ ]:
# =====================================================
# NLP SENTIMENT ANALYSIS USING IMDB DATASET
# =====================================================

import pandas as pd
import nltk
import re
import json

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

from textblob import TextBlob

# =====================================================
# DOWNLOAD REQUIRED NLTK PACKAGES
# =====================================================

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng') # Added this line
nltk.download('wordnet')
nltk.download('punkt_tab') # Added this line to fix the LookupError

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv("IMDB Dataset.csv", on_bad_lines='skip', engine='python')

print("Dataset Shape:", df.shape)

# =====================================================
# NLP PIPELINE
# =====================================================

def nlp_pipeline(text):

    # Lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword Removal
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # POS Tagging
    pos_tags = pos_tag(tokens)

    # Sentiment Analysis
    sentiment_score = TextBlob(
        " ".join(tokens)
    ).sentiment.polarity

    if sentiment_score > 0.1:
        sentiment = "Positive"
    elif sentiment_score < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return pd.Series([
        " ".join(tokens),
        pos_tags,
        sentiment_score,
        sentiment
    ])

# =====================================================
# APPLY NLP PIPELINE
# =====================================================

df[
    [
        "processed_review",
        "pos_tags",
        "sentiment_score",
        "predicted_sentiment"
    ]
] = df["review"].apply(nlp_pipeline)

# =====================================================
# DISPLAY SAMPLE OUTPUT
# =====================================================

print(
    df[
        [
            "review",
            "predicted_sentiment",
            "sentiment_score"
        ]
    ].head()
)

# =====================================================
# SAVE RESULTS AS JSON
# =====================================================

results = df[
    [
        "review",
        "processed_review",
        "predicted_sentiment",
        "sentiment_score"
    ]
].to_dict(orient="records")

with open(
    "imdb_sentiment_results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        results,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nJSON file saved successfully!")

# =====================================================
# CUSTOM REVIEW PREDICTION
# =====================================================

while True:

    review = input("\nEnter Review (type exit to stop): ")

    if review.lower() == "exit":
        break

    result = nlp_pipeline(review)

    print("\nPredicted Sentiment:", result[3])
    print("Sentiment Score:", round(result[2], 3))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Dataset Shape: (7088, 2)
                                              review predicted_sentiment  \
0  One of the other reviewers has mentioned that ...             Neutral   
1  A wonderful little production. <br /><br />The...            Positive   
2  I thought this was a wonderful way to spend ti...            Positive   
3  Basically there's a family where a little boy ...             Neutral   
4  Petter Mattei's "Love in the Time of Money" is...            Positive   

   sentiment_score  
0         0.006566  
1         0.235000  
2         0.347143  
3         0.064286  
4         0.193900  

JSON file saved successfully!
